# 02 - Commercial Cash Flow LGD Framework

This notebook is the parent **Commercial Cash Flow LGD Framework with sub-segments for term lending, revolving working-capital lines, receivables finance, contingent trade facilities, and asset finance**.

Commercial segments are separated because repayment source, EAD behaviour, collateral support, and workout path differ.


In [ ]:
import os
import sys

sys.path.insert(0, os.path.abspath('..'))

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from src.data_generation import generate_commercial_data
from src.commercial_data_controls import assign_framework_segment, run_commercial_data_controls
from src.lgd_calculation import CommercialLGDEngine, exposure_weighted_average
from src.validation import generate_validation_report, add_vintage_columns, build_vintage_lgd_summary
from src.reproducibility import set_global_seed

set_global_seed(43)

sns.set_theme(style='whitegrid', palette='muted')
pd.set_option('display.max_columns', 140)
pd.set_option('display.float_format', '{:.4f}'.format)

OUTPUT_DIR = os.path.abspath('../outputs')
TABLE_DIR = os.path.join(OUTPUT_DIR, 'tables')
FIG_DIR = os.path.join(OUTPUT_DIR, 'figures')
os.makedirs(TABLE_DIR, exist_ok=True)
os.makedirs(FIG_DIR, exist_ok=True)

SHOW_PLOTS = os.environ.get('LGD_NOTEBOOK_SHOW_PLOTS', '0') == '1'
print(f"Interactive plot display enabled: {SHOW_PLOTS}")


## Governance Baseline (Pre-Step-3)

- **Fallback hierarchy (standard policy):** observed workout inputs -> approved proxies -> conservative fallback with explicit disclosure.
- **Proxy logic:** synthetic borrower/workout data and proxy flags are used where internal default tapes are unavailable.
- **Discount-rate policy:** `discount_rate = max(contract_rate_proxy, cost_of_funds_proxy)`.
- **Calibration status:** portfolio-project baseline only; not internally calibrated to production workout history.
- **Output standard:** report base/economic LGD, downturn LGD, and regulatory-style final LGD using exposure-weighted aggregation.


## Objective
Build one umbrella commercial cash-flow LGD framework with sub-segment logic for term lending, revolving working-capital, receivables finance, contingent trade facilities, and asset/equipment finance.

## Drivers
- Term lending: leverage, DSCR/ICR, EBITDA margin proxy, security status, guarantee support
- Overdraft/revolver: drawdown behaviour, undrawn conversion, weaker-quality line risk
- Receivables: pool eligibility, ageing, dilution, debtor concentration, collections control
- Trade/contingent: claim conversion, security/cash support, tenor, contingent crystallisation
- Asset/equipment: asset type/age, residual exposure, repossession, remarketing liquidity

## Logic
- Discounted recovery-and-cost realised loss remains the base severity anchor (`realised_lgd`).
- Segment overlays produce base/economic LGD, downturn LGD, and regulatory-style final LGD.

## Downturn
- Stress is macro/driver-linked by segment (value and cashflow weakness, EAD conversion uplift, timing delay, liquidity deterioration).

## Validation
- Exposure-weighted outputs by segment and vintage, estimate-vs-realised checks, stability over time, and concentration views.

## Limitations
- Synthetic and proxy assumptions are used; production calibration needs internal workout histories and segment-specific governance approval.


## Segmentation Map (Parent Framework)

| Segment | Repayment Source | EAD Behaviour | Collateral / Support | Typical Workout Path |
|---|---|---|---|---|
| SME / middle-market term lending | Business cashflow + amortisation | Mostly drawn, low undrawn volatility | Property / PPSR / partial / unsecured + guarantees | Receivership, restructure, orderly asset realisation |
| Overdraft / revolver | Transactional cashflow + limit utilisation | High drawdown sensitivity near stress/default | Often mixed or weaker security packages | Freeze limits, collections sweep, restructure/write-down |
| Receivables / invoice finance | Receivables collections | Revolving with collateral turnover | PPSR receivables security | Ledger collections and dilution management |
| Trade / contingent facilities | Trade cycle cashflow / contingent conversion | Conversion risk from contingent to funded | Typically mixed contractual support | Conversion + recovery management |
| Asset / equipment finance | Operating cashflow + asset liquidation | Moderately revolving / amortising | PPSR equipment/asset security | Repossession and disposal |

Dedicated receivables reviewer notebook: `notebooks/04_receivables_invoice_finance_lgd.ipynb` (used to override receivables sub-segment metrics in this parent framework when available).
Dedicated trade/contingent reviewer notebook: `notebooks/05_trade_contingent_facilities_lgd.ipynb` (used to override trade/contingent proxy metrics in this parent framework when available).
Dedicated asset/equipment reviewer notebook: `notebooks/06_asset_equipment_finance_lgd.ipynb` (used to override asset/equipment metrics in this parent framework when available).


In [ ]:
# Load synthetic portfolio and add framework features
loans, cashflows = generate_commercial_data(n_loans=420, seed=43)

loans = loans.copy()
loans['leverage'] = pd.to_numeric(loans['leverage_ratio'], errors='coerce')
loans['icr'] = pd.to_numeric(loans['interest_coverage_ratio'], errors='coerce')
loans['dscr_proxy'] = (0.88 * loans['icr']).clip(0.30, 4.50)
loans['ebitda_margin_proxy'] = (loans['ebitda'] / loans['annual_revenue'].replace(0, np.nan)).fillna(0.0).clip(0.0, 0.60)
loans['industry_risk_bucket'] = loans['industry_risk_level'].fillna('Medium')

loans['guarantee_support_flag'] = loans['has_director_guarantee'].astype(int)
loans['covenant_breach_flag'] = (
    (loans['default_trigger'] == 'Covenant Breach')
    | ((loans['icr'] < 1.20) & (loans['leverage'] > 6.0))
).astype(int)
loans['watchlist_flag'] = (
    loans['default_trigger'].isin(['Covenant Breach', 'Voluntary Administration', 'Receivership'])
    | (loans['icr'] < 1.35)
    | (loans['leverage'] > 5.5)
).astype(int)

coverage_bins = [0.0, 0.50, 0.80, 1.00, 1.20, np.inf]
coverage_labels = ['<50%', '50-80%', '80-100%', '100-120%', '120%+']
loans['coverage_band'] = pd.cut(loans['security_coverage_ratio'], bins=coverage_bins, labels=coverage_labels, right=True)

size_bins = [0.0, 5e6, 20e6, 75e6, np.inf]
size_labels = ['Micro (<$5M)', 'Small ($5-20M)', 'Mid ($20-75M)', 'Large (>$75M)']
loans['borrower_size_band'] = pd.cut(loans['annual_revenue'], bins=size_bins, labels=size_labels, right=True)

loans['security_status'] = np.select(
    [
        (loans['security_coverage_ratio'] >= 1.00) & (loans['seniority'] == 'Senior Secured'),
        (loans['security_coverage_ratio'] >= 0.60) & (loans['seniority'] == 'Senior Secured'),
    ],
    ['Secured', 'Partially Secured'],
    default='Unsecured / Weakly Secured',
)

# Parent commercial framework segment (high-level map)
loans['framework_segment'] = assign_framework_segment(loans)

print('Loans:', loans.shape)
print('Cashflows:', cashflows.shape)
display(loans[['facility_type', 'framework_segment', 'security_type']].head(8))



In [ ]:
# Parent framework segment summary (counts and EAD)
segment_summary = (
    loans.groupby('framework_segment', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_share': g['ead'].sum() / loans['ead'].sum(),
                'mean_drawn_pct_limit': (g['drawn_balance'] / g['facility_limit']).mean(),
                'mean_security_coverage': g['security_coverage_ratio'].mean(),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('total_ead', ascending=False)
)

print('=== Parent Commercial Framework Segment Summary ===')
display(segment_summary.round(4))


## Term Lending Deep Dive

This segment focuses on amortising SME/middle-market term loans with explicit distinction between:
- secured term loans
- partially secured term loans
- unsecured/weakly secured term loans

Guarantee support is treated as a **secondary recovery support** (loss-reducing factor), not primary collateral.


In [ ]:
def weighted_by_dimension(df, dim_col, base_col, downturn_col, final_col, ead_col='ead'):
    rows = []
    for k, g in df.groupby(dim_col, observed=True):
        rows.append(
            {
                'dimension': dim_col,
                'bucket': str(k),
                'loan_count': len(g),
                'total_ead': g[ead_col].sum(),
                'ead_weighted_lgd_base': exposure_weighted_average(g, base_col, ead_col),
                'ead_weighted_lgd_downturn': exposure_weighted_average(g, downturn_col, ead_col),
                'ead_weighted_lgd_final': exposure_weighted_average(g, final_col, ead_col),
            }
        )
    return pd.DataFrame(rows)


term = loans[loans['facility_type'] == 'Term Loan'].copy()

industry_risk_factor = term['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)
security_risk_factor = term['security_status'].map({'Secured': 0.0, 'Partially Secured': 0.6, 'Unsecured / Weakly Secured': 1.2}).fillna(0.6)

guarantee_recovery_factor = np.where(term['guarantee_support_flag'] == 1, 0.94, 1.00)

term['lgd_base_economic'] = (
    (
        term['realised_lgd']
        + 0.035 * ((term['leverage'] - 4.0) / 4.0).clip(0.0, 1.5)
        + 0.040 * ((1.40 - term['dscr_proxy']) / 1.40).clip(0.0, 1.0)
        + 0.030 * ((1.60 - term['icr']) / 1.60).clip(0.0, 1.0)
        + 0.030 * ((0.15 - term['ebitda_margin_proxy']) / 0.15).clip(0.0, 1.0)
        + 0.025 * term['watchlist_flag']
        + 0.020 * term['covenant_breach_flag']
        + 0.020 * industry_risk_factor
        + 0.020 * security_risk_factor
    )
    * guarantee_recovery_factor
).clip(0.02, 0.98)

term['term_value_decline_proxy'] = (1.0 - term['security_coverage_ratio']).clip(0.0, 0.60)
term['term_cashflow_weakness'] = ((1.60 - term['icr']) / 1.60).clip(0.0, 1.0)
term['term_recovery_delay'] = ((term['workout_months'] - 18.0) / 24.0).clip(0.0, 1.0)

term['downturn_scalar'] = (
    1.00
    + 0.22 * term['term_value_decline_proxy']
    + 0.14 * term['term_cashflow_weakness']
    + 0.08 * term['term_recovery_delay']
).clip(1.00, 1.75)
term['lgd_downturn'] = (term['lgd_base_economic'] * term['downturn_scalar']).clip(0.0, 1.0)

term['moc_addon'] = 0.03 + 0.01 * term['watchlist_flag']
term['supervisory_floor_proxy'] = term['security_status'].map(
    {'Secured': 0.30, 'Partially Secured': 0.40, 'Unsecured / Weakly Secured': 0.50}
).fillna(0.45)
term['lgd_final_regulatory'] = np.maximum(term['lgd_downturn'] + term['moc_addon'], term['supervisory_floor_proxy']).clip(0.0, 1.0)

term_views = pd.concat(
    [
        weighted_by_dimension(term, 'security_status', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory'),
        weighted_by_dimension(term, 'coverage_band', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory'),
        weighted_by_dimension(term, 'borrower_size_band', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory'),
        weighted_by_dimension(term, 'industry_risk_bucket', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory'),
    ],
    ignore_index=True,
)

print('=== Term Lending Weighted LGD Outputs ===')
display(term_views.round(4))


## Overdraft / Revolver Deep Dive

Overdrafts and revolvers are more **EAD-sensitive** than term loans because undrawn commitment can convert near default.

This section adds explicit treatment for:
- current drawn amount
- undrawn commitment
- expected drawdown near default
- stress-state CCF uplift


In [ ]:
wc = loans[loans['facility_type'].isin(['Revolving Credit', 'Overdraft'])].copy()

wc['wc_sub_type'] = np.select(
    [
        (wc['facility_type'] == 'Revolving Credit') & (wc['security_type'].isin(['PPSR - Receivables', 'PPSR - Mixed'])),
        (wc['facility_type'] == 'Overdraft') & (wc['years_in_business'] >= 5) & (wc['watchlist_flag'] == 0),
        (wc['security_coverage_ratio'] < 0.55) & (wc['guarantee_support_flag'] == 0),
    ],
    [
        'Committed Working-Capital Line',
        'Overdraft with Transactional Evidence',
        'Unsecured WC - Weaker Quality',
    ],
    default='Other Revolver / Overdraft',
)

wc['drawn_pct_limit'] = (wc['drawn_balance'] / wc['facility_limit']).clip(0.0, 1.5)
wc['expected_drawdown_near_default'] = wc['undrawn_amount'] * wc['ccf']

weak_wc_flag = (wc['wc_sub_type'] == 'Unsecured WC - Weaker Quality').astype(int)
wc['stress_ccf'] = (wc['ccf'] + 0.08 + 0.10 * wc['watchlist_flag'] + 0.10 * weak_wc_flag).clip(0.40, 1.00)
wc['ead_stress'] = wc['drawn_balance'] + wc['stress_ccf'] * wc['undrawn_amount']
wc['ead_uplift_from_undrawn'] = (wc['ead_stress'] - wc['ead']).clip(lower=0.0)
wc['ead_uplift_pct'] = (wc['ead_uplift_from_undrawn'] / wc['ead'].replace(0, np.nan)).fillna(0.0).clip(0.0, 2.0)

wc_risk_factor = wc['industry_risk_bucket'].map({'Low': 0.0, 'Medium': 0.5, 'Elevated': 1.0}).fillna(0.5)

wc['lgd_base_economic'] = (
    (
        wc['realised_lgd']
        + 0.030 * ((wc['leverage'] - 4.5) / 4.5).clip(0.0, 1.5)
        + 0.030 * ((1.30 - wc['icr']) / 1.30).clip(0.0, 1.0)
        + 0.050 * weak_wc_flag
        + 0.040 * wc['watchlist_flag']
        + 0.030 * wc['ead_uplift_pct']
        + 0.025 * wc_risk_factor
    )
    * (1.0 - 0.03 * wc['guarantee_support_flag'])
).clip(0.03, 0.99)

wc['downturn_scalar'] = (
    1.00
    + 0.18 * wc['ead_uplift_pct']
    + 0.12 * ((1.40 - wc['icr']) / 1.40).clip(0.0, 1.0)
    + 0.08 * ((wc['workout_months'] - 16.0) / 20.0).clip(0.0, 1.0)
).clip(1.00, 1.80)
wc['lgd_downturn'] = (wc['lgd_base_economic'] * wc['downturn_scalar']).clip(0.0, 1.0)

wc_floor = {
    'Committed Working-Capital Line': 0.40,
    'Overdraft with Transactional Evidence': 0.42,
    'Unsecured WC - Weaker Quality': 0.60,
    'Other Revolver / Overdraft': 0.48,
}
wc['supervisory_floor_proxy'] = wc['wc_sub_type'].map(wc_floor).fillna(0.48)
wc['moc_addon'] = 0.035 + 0.015 * weak_wc_flag + 0.01 * wc['watchlist_flag']
wc['lgd_final_regulatory'] = np.maximum(wc['lgd_downturn'] + wc['moc_addon'], wc['supervisory_floor_proxy']).clip(0.0, 1.0)

wc_ead_ccf_summary = (
    wc.groupby('wc_sub_type', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead': g['ead'].sum(),
                'mean_drawn_pct_at_default': (g['drawn_balance'] / g['facility_limit']).mean(),
                'ead_weighted_ccf_current': exposure_weighted_average(g, 'ccf', 'ead'),
                'ead_weighted_ccf_stress': exposure_weighted_average(g, 'stress_ccf', 'ead'),
                'ead_weighted_uplift_pct': exposure_weighted_average(g, 'ead_uplift_pct', 'ead'),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('ead_weighted_lgd_final', ascending=False)
)

wc_weighted_facility = (
    wc.groupby(['facility_type', 'wc_sub_type'], observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base_economic', 'ead'),
                'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn', 'ead'),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_regulatory', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('ead_weighted_lgd_final', ascending=False)
)

print('=== Overdraft / Revolver EAD & CCF Summary ===')
display(wc_ead_ccf_summary.round(4))

print('=== Overdraft / Revolver Weighted LGD by Facility Type ===')
display(wc_weighted_facility.round(4))


## Integrated Base vs Downturn vs Regulatory Comparison

Term and overdraft/revolver use the deepened segment logic above.  
Receivables uses dedicated outputs from `04_receivables_invoice_finance_lgd.ipynb` where available.  
Trade/contingent uses dedicated outputs from `05_trade_contingent_facilities_lgd.ipynb` where available.  
Asset/equipment uses dedicated outputs from `06_asset_equipment_finance_lgd.ipynb` where available.

Segment limitations and expert judgement (concise):
- SME term lending: guarantee support and covenant impact are proxy-calibrated.
- Overdraft/revolver: CCF stress uplift is proxy-based and should be portfolio-calibrated.
- Receivables: ledger quality controls and dilution proxies require real collateral audit data.
- Trade/contingent: claim conversion and security support are simplified policy proxies.
- Asset/equipment: repossession and resale assumptions are synthetic by asset class.


In [ ]:
engine_input = loans.copy()
engine_input['icr'] = engine_input['interest_coverage_ratio']
engine = CommercialLGDEngine(moc=0.03)
generic = engine.apply_overlays(engine_input)

framework = loans[['loan_id', 'framework_segment', 'facility_type', 'security_type', 'industry', 'ead', 'default_date', 'realised_lgd']].copy()
framework = framework.merge(
    generic[['loan_id', 'lgd_base', 'lgd_downturn', 'lgd_final']],
    on='loan_id',
    how='left'
)

framework = framework.rename(
    columns={
        'lgd_base': 'lgd_base_framework',
        'lgd_downturn': 'lgd_downturn_framework',
        'lgd_final': 'lgd_final_framework',
    }
)

framework = framework.merge(
    term[['loan_id', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory']],
    on='loan_id',
    how='left',
    suffixes=('', '_term')
)
framework = framework.merge(
    wc[['loan_id', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory']],
    on='loan_id',
    how='left',
    suffixes=('', '_wc')
)

term_mask = framework['facility_type'].eq('Term Loan')
wc_mask = framework['facility_type'].isin(['Overdraft', 'Revolving Credit'])

framework.loc[term_mask, 'lgd_base_framework'] = framework.loc[term_mask, 'lgd_base_economic']
framework.loc[term_mask, 'lgd_downturn_framework'] = framework.loc[term_mask, 'lgd_downturn']
framework.loc[term_mask, 'lgd_final_framework'] = framework.loc[term_mask, 'lgd_final_regulatory']

framework.loc[wc_mask, 'lgd_base_framework'] = framework.loc[wc_mask, 'lgd_base_economic_wc']
framework.loc[wc_mask, 'lgd_downturn_framework'] = framework.loc[wc_mask, 'lgd_downturn_wc']
framework.loc[wc_mask, 'lgd_final_framework'] = framework.loc[wc_mask, 'lgd_final_regulatory_wc']

# Dedicated model overrides (if module notebooks have been executed)
framework['receivables_override_applied_flag'] = False
framework['trade_override_applied_flag'] = False
framework['asset_override_applied_flag'] = False

receivables_path = os.path.join(TABLE_DIR, 'receivables_invoice_finance_loan_level_output.csv')
if os.path.exists(receivables_path):
    recv_override = pd.read_csv(receivables_path)[['loan_id', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory']].rename(
        columns={
            'lgd_base_economic': 'lgd_base_receivables',
            'lgd_downturn': 'lgd_downturn_receivables',
            'lgd_final_regulatory': 'lgd_final_receivables',
        }
    )
    framework = framework.merge(recv_override, on='loan_id', how='left')
    recv_mask = framework['framework_segment'].eq('Receivables / Invoice Finance')
    has_override = recv_mask & framework['lgd_final_receivables'].notna()
    framework.loc[has_override, 'lgd_base_framework'] = framework.loc[has_override, 'lgd_base_receivables']
    framework.loc[has_override, 'lgd_downturn_framework'] = framework.loc[has_override, 'lgd_downturn_receivables']
    framework.loc[has_override, 'lgd_final_framework'] = framework.loc[has_override, 'lgd_final_receivables']
    framework.loc[has_override, 'receivables_override_applied_flag'] = True
    print(f"Receivables override rows applied: {int(has_override.sum())}")
else:
    print('Receivables override output not found; parent framework receivables row uses generic mapping.')

trade_path = os.path.join(TABLE_DIR, 'trade_contingent_loan_level_output.csv')
if os.path.exists(trade_path):
    trade_override = pd.read_csv(trade_path)[['loan_id', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory']].rename(
        columns={
            'lgd_base_economic': 'lgd_base_trade',
            'lgd_downturn': 'lgd_downturn_trade',
            'lgd_final_regulatory': 'lgd_final_trade',
        }
    )
    framework = framework.merge(trade_override, on='loan_id', how='left')
    trade_mask = framework['framework_segment'].eq('Trade / Contingent Facilities (Proxy)')
    has_trade_override = trade_mask & framework['lgd_final_trade'].notna()
    framework.loc[has_trade_override, 'lgd_base_framework'] = framework.loc[has_trade_override, 'lgd_base_trade']
    framework.loc[has_trade_override, 'lgd_downturn_framework'] = framework.loc[has_trade_override, 'lgd_downturn_trade']
    framework.loc[has_trade_override, 'lgd_final_framework'] = framework.loc[has_trade_override, 'lgd_final_trade']
    framework.loc[has_trade_override, 'trade_override_applied_flag'] = True
    print(f"Trade/contingent override rows applied: {int(has_trade_override.sum())}")
else:
    print('Trade/contingent override output not found; parent framework trade row uses proxy mapping.')

asset_path = os.path.join(TABLE_DIR, 'asset_equipment_finance_loan_level_output.csv')
if os.path.exists(asset_path):
    asset_override = pd.read_csv(asset_path)[['loan_id', 'lgd_base_economic', 'lgd_downturn', 'lgd_final_regulatory']].rename(
        columns={
            'lgd_base_economic': 'lgd_base_asset',
            'lgd_downturn': 'lgd_downturn_asset',
            'lgd_final_regulatory': 'lgd_final_asset',
        }
    )
    framework = framework.merge(asset_override, on='loan_id', how='left')
    asset_mask = framework['framework_segment'].eq('Asset / Equipment Finance')
    has_asset_override = asset_mask & framework['lgd_final_asset'].notna()
    framework.loc[has_asset_override, 'lgd_base_framework'] = framework.loc[has_asset_override, 'lgd_base_asset']
    framework.loc[has_asset_override, 'lgd_downturn_framework'] = framework.loc[has_asset_override, 'lgd_downturn_asset']
    framework.loc[has_asset_override, 'lgd_final_framework'] = framework.loc[has_asset_override, 'lgd_final_asset']
    framework.loc[has_asset_override, 'asset_override_applied_flag'] = True
    print(f"Asset/equipment override rows applied: {int(has_asset_override.sum())}")
else:
    print('Asset/equipment override output not found; parent framework asset row uses generic/term mapping.')

weighted_lgd_by_segment = (
    framework.groupby('framework_segment', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_lgd_base': exposure_weighted_average(g, 'lgd_base_framework', 'ead'),
                'ead_weighted_lgd_downturn': exposure_weighted_average(g, 'lgd_downturn_framework', 'ead'),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_framework', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values('ead_weighted_lgd_final', ascending=False)
)

base_vs_downturn = weighted_lgd_by_segment.copy()
base_vs_downturn['downturn_sensitivity_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_downturn'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)
base_vs_downturn['final_minus_base_pp'] = (
    (base_vs_downturn['ead_weighted_lgd_final'] - base_vs_downturn['ead_weighted_lgd_base']) * 100
)

print('=== Weighted LGD by Framework Segment ===')
display(weighted_lgd_by_segment.round(4))

print('=== Base vs Downturn Comparison ===')
display(base_vs_downturn.round(4))


In [ ]:
# Monitoring, vintage, concentration, and validation outputs
framework = add_vintage_columns(framework, date_col='default_date', fallback_years_on_book=3.0)

monitoring_by_year = (
    framework.groupby(['framework_segment', 'default_year'], observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_framework', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)

weighted_lgd_by_vintage = build_vintage_lgd_summary(
    framework.copy(),
    actual_col='realised_lgd',
    predicted_col='lgd_final_framework',
    ead_col='ead',
    vintage_col='origination_year',
    default_year_col='default_year',
)

weighted_lgd_by_segment_vintage = (
    framework.dropna(subset=['origination_year'])
    .groupby(['framework_segment', 'origination_year'], observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_framework', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
    .sort_values(['framework_segment', 'origination_year'])
)

estimate_vs_realised_by_segment = (
    framework.groupby('framework_segment', observed=True)
    .apply(
        lambda g: pd.Series(
            {
                'loan_count': len(g),
                'total_ead': g['ead'].sum(),
                'ead_weighted_realised_lgd': exposure_weighted_average(g, 'realised_lgd', 'ead'),
                'ead_weighted_estimated_lgd': exposure_weighted_average(g, 'lgd_final_framework', 'ead'),
            }
        ),
        include_groups=False,
    )
    .reset_index()
)
estimate_vs_realised_by_segment['estimated_minus_realised_pp'] = (
    (estimate_vs_realised_by_segment['ead_weighted_estimated_lgd'] - estimate_vs_realised_by_segment['ead_weighted_realised_lgd']) * 100
)

stability_checks_by_segment = (
    monitoring_by_year.groupby('framework_segment', observed=True)
    .agg(
        years_observed=('default_year', 'nunique'),
        mean_weighted_lgd_final=('ead_weighted_lgd_final', 'mean'),
        std_weighted_lgd_final=('ead_weighted_lgd_final', 'std'),
        min_weighted_lgd_final=('ead_weighted_lgd_final', 'min'),
        max_weighted_lgd_final=('ead_weighted_lgd_final', 'max'),
    )
    .reset_index()
)
stability_checks_by_segment['range_weighted_lgd_final'] = (
    stability_checks_by_segment['max_weighted_lgd_final'] - stability_checks_by_segment['min_weighted_lgd_final']
)

def concentration_view(df, col):
    out = (
        df.groupby(col, observed=True)
        .apply(
            lambda g: pd.Series(
                {
                    'loan_count': len(g),
                    'total_ead': g['ead'].sum(),
                    'ead_weighted_lgd_final': exposure_weighted_average(g, 'lgd_final_framework', 'ead'),
                }
            ),
            include_groups=False,
        )
        .reset_index()
        .sort_values('total_ead', ascending=False)
    )
    out['ead_share'] = out['total_ead'] / max(df['ead'].sum(), 1.0)
    return out

concentration_by_industry = concentration_view(framework, 'industry')
concentration_by_facility_type = concentration_view(framework, 'facility_type')
concentration_by_security_type = concentration_view(framework, 'security_type')

core_validation_checks = pd.DataFrame(
    [
        {'check_name': 'framework_base_between_0_and_1', 'passed': framework['lgd_base_framework'].between(0, 1).all(), 'failed_rows': int((~framework['lgd_base_framework'].between(0, 1)).sum()), 'detail': 'lgd_base_framework in [0,1]'},
        {'check_name': 'framework_downturn_between_0_and_1', 'passed': framework['lgd_downturn_framework'].between(0, 1).all(), 'failed_rows': int((~framework['lgd_downturn_framework'].between(0, 1)).sum()), 'detail': 'lgd_downturn_framework in [0,1]'},
        {'check_name': 'framework_final_between_0_and_1', 'passed': framework['lgd_final_framework'].between(0, 1).all(), 'failed_rows': int((~framework['lgd_final_framework'].between(0, 1)).sum()), 'detail': 'lgd_final_framework in [0,1]'},
        {'check_name': 'downturn_not_below_base', 'passed': (framework['lgd_downturn_framework'] >= framework['lgd_base_framework']).all(), 'failed_rows': int((framework['lgd_downturn_framework'] < framework['lgd_base_framework']).sum()), 'detail': 'downturn >= base'},
        {'check_name': 'final_not_below_downturn', 'passed': (framework['lgd_final_framework'] >= framework['lgd_downturn_framework']).all(), 'failed_rows': int((framework['lgd_final_framework'] < framework['lgd_downturn_framework']).sum()), 'detail': 'final >= downturn'},
    ]
)

framework_data_controls = run_commercial_data_controls(loans, cashflows, segment_col='framework_segment')
validation_checks = pd.concat([core_validation_checks, framework_data_controls], ignore_index=True)

val_report = generate_validation_report(
    framework.dropna(subset=['lgd_final_framework']).copy(),
    actual_col='realised_lgd',
    predicted_col='lgd_final_framework',
    segment_col='framework_segment',
    date_col='default_date',
)

print('=== Validation Checks ===')
display(validation_checks)
print('=== Estimate vs Realised (Weighted) ===')
display(estimate_vs_realised_by_segment.round(4))
print('=== Accuracy Snapshot ===')
print(val_report['accuracy'])


In [ ]:
# Charts
plot_df = weighted_lgd_by_segment.set_index('framework_segment')[
    ['ead_weighted_lgd_base', 'ead_weighted_lgd_downturn', 'ead_weighted_lgd_final']
]
fig, ax = plt.subplots(figsize=(11, 5))
plot_df.plot(kind='bar', ax=ax, color=['#4c72b0', '#dd8452', '#c44e52'])
ax.set_title('Commercial Framework: Weighted LGD by Segment (Base/Downturn/Final)')
ax.set_ylabel('LGD')
ax.set_xlabel('Segment')
plt.xticks(rotation=15, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'commercial_framework_weighted_lgd_by_segment.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(9, 5))
ax.bar(wc_ead_ccf_summary['wc_sub_type'], wc_ead_ccf_summary['ead_weighted_uplift_pct'], color='#55a868')
ax.set_title('Overdraft / Revolver: EAD Uplift from Undrawn Commitment (Weighted)')
ax.set_ylabel('EAD uplift %')
ax.set_xlabel('WC Sub-type')
plt.xticks(rotation=20, ha='right')
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'commercial_framework_ead_ccf_uplift.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)

fig, ax = plt.subplots(figsize=(11, 5))
for seg, g in monitoring_by_year.groupby('framework_segment', observed=True):
    ax.plot(g['default_year'], g['ead_weighted_lgd_final'], marker='o', label=seg)
ax.set_title('Commercial Framework Monitoring: Weighted Final LGD Over Time')
ax.set_xlabel('Default Year')
ax.set_ylabel('EAD-weighted Final LGD')
ax.legend(fontsize=8, ncol=2)
plt.tight_layout()
plt.savefig(os.path.join(FIG_DIR, 'commercial_framework_monitoring_over_time.png'), dpi=140, bbox_inches='tight')
if SHOW_PLOTS:
    plt.show()
else:
    plt.close(fig)


In [ ]:
# Exports
framework_export = framework[
    [
        'loan_id', 'framework_segment', 'facility_type', 'security_type', 'industry', 'ead', 'realised_lgd',
        'origination_year', 'default_year', 'origination_year_source',
        'lgd_base_framework', 'lgd_downturn_framework', 'lgd_final_framework'
    ]
].copy()

framework_export.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_loan_level_output.csv'), index=False)
segment_summary.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_segment_summary.csv'), index=False)
weighted_lgd_by_segment.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_weighted_lgd_by_segment.csv'), index=False)
term_views.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_term_weighted_lgd_views.csv'), index=False)
wc_ead_ccf_summary.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_ead_ccf_summary.csv'), index=False)
wc_weighted_facility.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_wc_weighted_lgd_by_facility.csv'), index=False)
base_vs_downturn.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_base_vs_downturn_comparison.csv'), index=False)
monitoring_by_year.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_monitoring_by_year.csv'), index=False)
weighted_lgd_by_vintage.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_weighted_lgd_by_vintage.csv'), index=False)
weighted_lgd_by_segment_vintage.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_weighted_lgd_by_segment_vintage.csv'), index=False)
estimate_vs_realised_by_segment.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_estimate_vs_realised_by_segment.csv'), index=False)
stability_checks_by_segment.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_stability_checks.csv'), index=False)
concentration_by_industry.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_concentration_by_industry.csv'), index=False)
concentration_by_facility_type.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_concentration_by_facility_type.csv'), index=False)
concentration_by_security_type.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_concentration_by_security_type.csv'), index=False)
validation_checks.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_validation_checks.csv'), index=False)

segment_override_summary = pd.DataFrame([
    {
        'receivables_segment_rows': int((framework['framework_segment'] == 'Receivables / Invoice Finance').sum()),
        'receivables_override_rows_applied': int(framework.get('receivables_override_applied_flag', pd.Series(False, index=framework.index)).sum()),
        'trade_segment_rows': int((framework['framework_segment'] == 'Trade / Contingent Facilities (Proxy)').sum()),
        'trade_override_rows_applied': int(framework.get('trade_override_applied_flag', pd.Series(False, index=framework.index)).sum()),
        'asset_segment_rows': int((framework['framework_segment'] == 'Asset / Equipment Finance').sum()),
        'asset_override_rows_applied': int(framework.get('asset_override_applied_flag', pd.Series(False, index=framework.index)).sum()),
    }
])
segment_override_summary.to_csv(os.path.join(TABLE_DIR, 'commercial_framework_segment_override_summary.csv'), index=False)

segment_override_summary[[
    'receivables_segment_rows', 'receivables_override_rows_applied'
]].rename(columns={'receivables_override_rows_applied': 'override_rows_applied'}).to_csv(
    os.path.join(TABLE_DIR, 'commercial_framework_receivables_override_summary.csv'), index=False
)


print('Saved commercial framework outputs:')
print('- commercial_framework_loan_level_output.csv')
print('- commercial_framework_segment_summary.csv')
print('- commercial_framework_weighted_lgd_by_segment.csv')
print('- commercial_framework_term_weighted_lgd_views.csv')
print('- commercial_framework_ead_ccf_summary.csv')
print('- commercial_framework_wc_weighted_lgd_by_facility.csv')
print('- commercial_framework_base_vs_downturn_comparison.csv')
print('- commercial_framework_monitoring_by_year.csv')
print('- commercial_framework_weighted_lgd_by_vintage.csv')
print('- commercial_framework_weighted_lgd_by_segment_vintage.csv')
print('- commercial_framework_estimate_vs_realised_by_segment.csv')
print('- commercial_framework_stability_checks.csv')
print('- commercial_framework_concentration_by_industry.csv')
print('- commercial_framework_concentration_by_facility_type.csv')
print('- commercial_framework_concentration_by_security_type.csv')
print('- commercial_framework_validation_checks.csv')
print('- commercial_framework_segment_override_summary.csv')
print('- commercial_framework_receivables_override_summary.csv')
